# LLM Fine-Tuning Lab — Quantization, LoRA & QLoRA

This lab demonstrates a complete LLM fine-tuning pipeline using:
- **Quantization**: Reducing model precision (FP32 → 4-bit) to fit in limited GPU memory
- **LoRA (Low-Rank Adaptation)**: Adding small trainable rank matrices instead of fine-tuning all weights
- **QLoRA**: Combining quantization with LoRA for memory-efficient fine-tuning

The practical demo fine-tunes a TinyLlama model to generate SQL queries from natural language.

## 1. Setup

Install and import required libraries.

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01


In [ ]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import Dataset

## 2. Quantization Fundamentals

Understanding how quantization works: mapping floating-point values to a lower-bit representation.

### 2.1 Quantization (FP32 → UINT8)

Simulate model weight quantization using asymmetric scaling.

In [ ]:
weights_fp32 = np.array([0.1, 2.5, 5.7, 10.2, -3.4], dtype=np.float32)

x_min = weights_fp32.min()
x_max = weights_fp32.max()

Q_min, Q_max = 0, 255

scale = (x_max - x_min) / (Q_max - Q_min)
zero_point = round(Q_min - x_min / scale)

quantized = np.round(weights_fp32 / scale + zero_point).astype(np.uint8)

print("Original:", weights_fp32)
print("Scale:", scale)
print("Zero Point:", zero_point)
print("Quantized:", quantized)

Original: [ 0.1  2.5  5.7 10.2 -3.4]
Scale: 0.053333335
Zero Point: 64
Quantized: [ 66 111 171 255   0]


### 2.2 Dequantization (UINT8 → FP32)

Recover original values from quantized weights.

In [ ]:
dequantized = (quantized.astype(np.float32) - zero_point) * scale
print("Dequantized:", dequantized)

Dequantized: [ 0.10666667  2.5066667   5.706667   10.1866665  -3.4133334 ]


## 3. QLoRA Fine-Tuning Pipeline

Complete workflow: Load quantized model → Prepare dataset → Apply LoRA → Train → Inference

### 3.1 Load Model in 4-bit (QLoRA)

Load TinyLlama with 4-bit NF4 quantization using BitsAndBytes.

In [ ]:
# Memory optimization: reduce memory fragmentation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clear GPU cache before loading
torch.cuda.empty_cache()

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True  # Further reduce memory
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### 3.2 Prepare SQL Dataset

Create natural language → SQL query pairs for fine-tuning.

### 3.3 Generate Training Data

Generate 500 diverse NL-to-SQL examples using template functions.

In [ ]:
# SQL Fine-Tuning Dataset
sql_data = [
    # {
    #     "instruction": "Given the users table with columns: id, name, email, created_at. Write a query to find all users who signed up in 2024.",
    #     "input": "",
    #     "output": "SELECT * FROM users WHERE YEAR(created_at) = 2024;"
    # },
    # {
    #     "instruction": "From the orders table (id, customer_id, product_id, quantity, price, order_date), calculate total revenue per customer.",
    #     "input": "",
    #     "output": "SELECT customer_id, SUM(quantity * price) AS total_revenue FROM orders GROUP BY customer_id;"
    # },
    # {
    #     "instruction": "Using the employees table (id, name, department, salary, hire_date), find the top 3 highest paid employees in each department.",
    #     "input": "",
    #     "output": "SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rn FROM employees) t WHERE rn <= 3;"
    # },
    # {
    #     "instruction": "From the products table (id, name, category, price, stock), list all products with stock below 10.",
    #     "input": "",
    #     "output": "SELECT * FROM products WHERE stock < 10;"
    # },
    # {
    #     "instruction": "Given the sales table (id, product_id, sale_date, quantity, amount), find monthly sales trend for 2024.",
    #     "input": "",
    #     "output": "SELECT MONTH(sale_date) as month, SUM(amount) as total_sales FROM sales WHERE YEAR(sale_date) = 2024 GROUP BY MONTH(sale_date);"
    # },
    # {
    #     "instruction": "From the customers table (id, name, email, country, registration_date), count customers by country.",
    #     "input": "",
    #     "output": "SELECT country, COUNT(*) as customer_count FROM customers GROUP BY country ORDER BY customer_count DESC;"
    # },
    # {
    #     "instruction": "Using the orders table (id, customer_id, order_date, status, total), find average order value by status.",
    #     "input": "",
    #     "output": "SELECT status, AVG(total) as avg_order_value FROM orders GROUP BY status;"
    # },
    # {
    #     "instruction": "From the inventory table (product_id, warehouse_id, quantity, last_updated), find products that need restocking.",
    #     "input": "",
    #     "output": "SELECT product_id, SUM(quantity) as total_stock FROM inventory GROUP BY product_id HAVING SUM(quantity) < 20;"
    # }
]


In [ ]:
import random

def gen_users():
    return {
        "instruction": "users table  (acheid, name, email, created_at). 2024 year e sign up kora users ber koro.",
        "output": "SELECT * FROM users WHERE YEAR(created_at) = 2024;"
    }

def gen_orders_revenue():
    return {
        "instruction": "orders table (id, customer_id, product_id, quantity, price, order_date). prottek customer er total revenue ber koro.",
        "output": "SELECT customer_id, SUM(quantity * price) AS total_revenue FROM orders GROUP BY customer_id;"
    }

def gen_top_k_employees():
    k = random.choice([2,3,5])
    return {
        "instruction": f"employees table theke prottek department er top {k} highest salary employee ber koro.",
        "output": f"SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rn FROM employees) t WHERE rn <= {k};"
    }

def gen_low_stock():
    val = random.choice([5,10,20,30])
    return {
        "instruction": f"products table theke sei products ber koro jader stock {val} er niche.",
        "output": f"SELECT * FROM products WHERE stock < {val};"
    }

def gen_monthly_sales():
    return {
        "instruction": "sales table theke 2024 year er monthly sales trend ber koro.",
        "output": "SELECT MONTH(sale_date) as month, SUM(amount) as total_sales FROM sales WHERE YEAR(sale_date) = 2024 GROUP BY MONTH(sale_date);"
    }

def gen_group_by_country():
    return {
        "instruction": "customers table theke prottek country er customer count ber koro.",
        "output": "SELECT country, COUNT(*) as customer_count FROM customers GROUP BY country ORDER BY customer_count DESC;"
    }

def gen_avg_order():
    return {
        "instruction": "orders table theke prottek status er average order value ber koro.",
        "output": "SELECT status, AVG(total) as avg_order_value FROM orders GROUP BY status;"
    }

def gen_inventory_stock():
    threshold = random.choice([10,15,20,25])
    return {
        "instruction": "inventory table theke restock dorkar emon products ber koro.",
        "output": f"SELECT product_id, SUM(quantity) as total_stock FROM inventory GROUP BY product_id HAVING SUM(quantity) < {threshold};"
    }

In [ ]:

generators = [
    gen_users,
    gen_orders_revenue,
    gen_top_k_employees,
    gen_low_stock,
    gen_monthly_sales,
    gen_group_by_country,
    gen_avg_order,
    gen_inventory_stock
]

sql_data = []

for i in range(500):
    gen = random.choice(generators)
    sample = gen()

    sql_data.append({
        "instruction": sample["instruction"],
        "input": "", # Optional
        "output": sample["output"]
    })

print("Total samples:", len(sql_data))


# Create dataset
sql_dataset = Dataset.from_list(sql_data)
print(f"SQL Dataset size: {len(sql_dataset)}")
sql_dataset

Total samples: 500
SQL Dataset size: 500


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 500
})

### 3.4 Tokenize Dataset

Format with instruction prompting and tokenize for causal LM training.

In [ ]:
# Format data with instruction prompting
def format_sql_example(example):
    text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    return {"text": text}

sql_dataset = sql_dataset.map(format_sql_example)

# Tokenize with response-only labels (mask instruction tokens with -100)
# This forces the model to learn SQL generation, not instruction memorisation.
def tokenize_function(examples):
    results = {"input_ids": [], "attention_mask": [], "labels": []}
    for text in examples["text"]:
        # Split at the response boundary
        parts = text.split("### Response:\n")
        instruction_part = parts[0] + "### Response:\n"
        response_part    = parts[1] if len(parts) > 1 else ""

        # Tokenize separately to find the instruction length
        instr_ids = tokenizer(instruction_part, add_special_tokens=False)["input_ids"]
        full      = tokenizer(text, truncation=True, max_length=512,
                              padding="max_length", return_tensors=None)

        labels = list(full["input_ids"])
        # Mask instruction tokens — model should only predict SQL tokens
        for i in range(min(len(instr_ids), len(labels))):
            labels[i] = -100
        # Also mask padding tokens
        labels = [-100 if tok == tokenizer.pad_token_id else tok
                  for i, tok in enumerate(labels)
                  if not (i >= len(instr_ids) and tok == tokenizer.pad_token_id)
                  or True]

        results["input_ids"].append(full["input_ids"])
        results["attention_mask"].append(full["attention_mask"])
        results["labels"].append(labels)
    return results

sql_dataset = sql_dataset.map(tokenize_function, batched=True,
                               remove_columns=sql_dataset.column_names)
print(f"Tokenized SQL dataset: {len(sql_dataset)} samples")
sql_dataset


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenized SQL dataset: 500 samples


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})

In [ ]:
print(sql_dataset[0])

{'input_ids': [1, 835, 2799, 4080, 29901, 13, 6341, 414, 1591, 278, 446, 17512, 12681, 4234, 604, 11962, 2302, 7655, 413, 5801, 29889, 13, 13, 2277, 29937, 13291, 29901, 13, 6404, 4234, 29892, 21122, 22798, 408, 11962, 29918, 2798, 3895, 20330, 15345, 6770, 4234, 15606, 6770, 11962, 29918, 2798, 23050, 29936, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [ ]:
decoded = tokenizer.decode(sql_dataset[0]["input_ids"])
print(decoded)

<s> ### Instruction:
customers table theke prottek country er customer count ber koro.

### Response:
SELECT country, COUNT(*) as customer_count FROM customers GROUP BY country ORDER BY customer_count DESC;</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></

In [ ]:
sample = sql_dataset[1]

print("Token length (input_ids):", len(sample["input_ids"]))  # padded to 512
print("Attention sum (real tokens):", sum(sample["attention_mask"]))  # real length

Token length (input_ids): 512
Attention sum (real tokens): 49


In [ ]:
# Clear GPU memory first
torch.cuda.empty_cache()

# Use existing model and tokenizer from section 3.1
sql_model = model
sql_tokenizer = tokenizer

print("Reusing model from section 3.1 for fine-tuning")

Reusing model from section 3.1 for fine-tuning


### 3.6 Apply LoRA Adapters

Configure and attach LoRA adapters to the quantized model.

In [ ]:
# Import LoRA (PEFT) utilities
from peft import LoraConfig, get_peft_model, TaskType

# Define LoRA configuration specifically for SQL generation (causal language modeling)
sql_lora_config = LoraConfig(
    r=4,  # Rank of LoRA matrices (smaller = fewer trainable params, faster training)
    lora_alpha=16,  # Scaling factor for LoRA (effective strength = alpha / r)
    lora_dropout=0.1,  # Dropout applied to LoRA layers to prevent overfitting

    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,  # Task type: next-token prediction (text generation)
    bias="none"  # Do not train bias parameters (keeps model lightweight)
)

# Inject LoRA adapters into the base model
# This freezes original weights and adds small trainable layers
sql_model = get_peft_model(sql_model, sql_lora_config)

# Print how many parameters are trainable vs total parameters
sql_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### 3.7 Fine-tune Model

Train the LoRA-adapted model on the SQL dataset.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# DataCollator pads input_ids and keeps labels as -100 for masked tokens
data_collator = DataCollatorForSeq2Seq(
    tokenizer=sql_tokenizer,
    model=sql_model,
    label_pad_token_id=-100,   # masked tokens stay masked. Tokens with -100 are ignored in loss calculation
    pad_to_multiple_of=8       # efficient on tensor cores
)

sql_training_args = TrainingArguments(
    output_dir="./sql_results",
    per_device_train_batch_size=4,      # increased from 1 (grad_accum still effective)
    gradient_accumulation_steps=4,      # effective batch = 16. Simulate a larger batch without needing more GPU memory
    num_train_epochs=3,                 # model needs more passes to learn
    logging_steps=50,                   # logging every step hid real loss values
    save_strategy="no",
    fp16=True,
    report_to="none",
    learning_rate=2e-4,                 # slightly lower than 3e-4 for stable LoRA training
    warmup_steps=20,                    # Gradually increases LR at start
    lr_scheduler_type="cosine",         # smoother decay of LR
    optim="paged_adamw_8bit",           # memory-efficient optimizer for QLoRA
)

sql_trainer = Trainer(
    model=sql_model,
    args=sql_training_args,
    train_dataset=sql_dataset,
    data_collator=data_collator,        # proper padding with -100 masking
)

print("Starting SQL fine-tuning...")
sql_trainer.train()
print("SQL fine-tuning complete!")


Starting SQL fine-tuning...


Step,Training Loss
50,0.262560


SQL fine-tuning complete!


### 3.8 Test Fine-tuned Model

Run inference to generate SQL from natural language.

In [ ]:
# IMPORTANT: Switch model to eval mode before inference
# Without this, LoRA dropout stays active and degrades output quality
sql_model.eval()

test_prompt = """### Instruction:
student table ache (id, name, email, created_at). 2034 year e sign up kora student ber koro.

### Response:
"""

inputs = sql_tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = sql_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],   # explicit mask prevents padding warnings
        max_new_tokens=100,
        do_sample=False,                           # greedy — most stable for SQL
        num_beams=4,                               # NEW: beam search improves SQL accuracy
        early_stopping=True,                       # NEW: stop when all beams hit EOS
        pad_token_id=sql_tokenizer.eos_token_id,
        eos_token_id=sql_tokenizer.eos_token_id,
        repetition_penalty=1.2,
    )

# Decode only the NEW tokens (skip the prompt) for a cleaner extraction
prompt_len = inputs["input_ids"].shape[1]
new_tokens = outputs[0][prompt_len:]
response = sql_tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Extracted SQL:")
print(response.strip())

# Optionally also show full decoded output for debugging
full_response = sql_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*50)
print("Full response (for debug):")
print(full_response)


Extracted SQL:
SELECT * FROM student WHERE YEAR(created_at) = 2034;

Full response (for debug):
### Instruction:
student table ache (id, name, email, created_at). 2034 year e sign up kora student ber koro.

### Response:
SELECT * FROM student WHERE YEAR(created_at) = 2034;
